# Claim type and core claim test v02 (propositions)

Changes from v01:

- Unit is the **proposition** (`P009`), not the atom (`P009-2`).
- Each model sees the **whole document** and the full list of propositions; each proposition is judged as a whole.
- New types: `ANALYSIS` (working through facts, not a subjective view) and `OBSERVATION` (objective truth, replaces v01 `Fact`).

Types: OBSERVATION, ANALYSIS, VIEW, FORECAST, PREDICTION, ASSUMPTION. Plus `core` TRUE/FALSE.

Raw responses are cached in `outputs/claim_core_test_v02/`; delete a file to re-run that model.

In [ ]:
import json
import re
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from incolor.argumentation.backends.claude_agent import ClaudeAgentBackend

OUT = ROOT / "outputs" / "claim_core_test_v02"
OUT.mkdir(parents=True, exist_ok=True)
MODELS = ["haiku", "sonnet", "opus"]
TYPES = ["OBSERVATION", "ANALYSIS", "VIEW", "FORECAST", "PREDICTION", "ASSUMPTION"]

DOC = re.sub(r"\A---.*?---\s*", "", (ROOT / "samples/BofA_Sept_FOMC_-_hike_or_bond_spike.txt").read_text(encoding="utf-8"), flags=re.S)
props = {
    m.group(1): m.group(2)
    for ln in (ROOT / "outputs/atomise_v01/opus/input.txt").read_text(encoding="utf-8").splitlines()
    if (m := re.match(r"\[(P\d+)\]\s+(.*)", ln))
}
print(len(props), "propositions,", len(DOC), "chars of document")

120 propositions, 13792 chars of document


In [2]:
SYSTEM = """You annotate propositions extracted from a sell-side FX/rates research note. You get the FULL note, then the list of propositions (each is a sentence or clause of the note, lightly normalised). Use the whole note to understand each proposition's role, but judge every proposition as a whole.

For EACH proposition give two independent judgements.

1. claim_type - exactly one of:
- OBSERVATION: a statement of objective truth - what is or was the case and can be checked without the author's judgement: published data, market pricing, history, what someone said or did, scheduled events, definitions, and bare declaratives (including bare statements about what a publication will contain, e.g. "SEP is 50bps total hikes in '26"). Expectations or forecasts attributed to the market or third parties ("widely expected", "priced") are OBSERVATIONs about those parties.
- ANALYSIS: the author working through facts without taking a subjective stance: a comparison, calculation, decomposition, mechanical implication, or reading of what the data or pricing show or imply (e.g. what pricing implies against history, how a figure compares with a threshold). Usually built on observations; not an expectation about the future and not an opinion.
- VIEW: the author's own subjective judgement, interpretation, causal attribution, characterisation or risk assessment, including forward-looking judgements that lack a measurable value, horizon or identifiable event.
- FORECAST: the author's own expectation of a quantitatively measurable outcome with a value AND a horizon (levels, ranges or changes in yields, prices, rates, macro data, projections).
- PREDICTION: the author's own expectation that a specific, identifiable future event will or will not happen (a policy decision, dissent, statement, announcement). It must be the author's own thought, marked by author-voice or expectation cues (the author expects/thinks/guesses, likely, should, will, may). A bare declarative with no such cue is an OBSERVATION.
- ASSUMPTION: a proposition that is only a supposed condition or scenario header, not asserted by the author.
For a conditional ("if X then Y"), type the conclusion Y as the author's claim under that condition.

2. core - true or false. Core claims are the author's headline conclusions: the few main takeaways the whole note exists to deliver and that the rest of the note supports. False for supporting evidence, data, premises, background, caveats, secondary details, scenario branches, and restatements of lower-level detail, and false for OBSERVATION, ANALYSIS and ASSUMPTION unless it is itself the thesis. Be selective: a note of this length has only a handful of core claims, so when unsure return false. Where a conclusion is restated in several places, mark only the proposition that states it in its most general, headline form.

Return ONLY a JSON array, one object per proposition, in input order:
{"id": "P005", "claim_type": "PREDICTION", "core": false, "why": "<=8 words"}
No text outside the JSON."""

USER = f"<document>\n{DOC}\n</document>\n\n<propositions>\n" + "\n".join(f"[{k}] {v}" for k, v in props.items()) + "\n</propositions>"


def parse(raw: str) -> list[dict]:
    return json.loads(raw[raw.index("[") : raw.rindex("]") + 1])


def run(model: str) -> dict:
    path = OUT / f"{model}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    backend = ClaudeAgentBackend(model=model)
    raw = backend.generate(SYSTEM, USER, {})
    result = {"raw": raw, "rows": parse(raw), "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


with ThreadPoolExecutor(len(MODELS)) as pool:
    results = dict(zip(MODELS, pool.map(run, MODELS)))

for m, r in results.items():
    cost = sum(u.get("cost_usd") or 0 for u in r["usage"])
    print(f"{m}: {len(r['rows'])} rows, ${cost:.3f}")

haiku: 120 rows, $0.010
sonnet: 120 rows, $0.101
opus: 120 rows, $0.245


In [3]:
df = pd.DataFrame({"text": props})
for m, r in results.items():
    by_id = {row["id"]: row for row in r["rows"]}
    df[f"{m}_type"] = [
        (by_id[i].get("claim_type") or "?").strip().upper() if i in by_id else "MISSING" for i in df.index
    ]
    df[f"{m}_core"] = [bool(by_id[i].get("core")) if i in by_id else False for i in df.index]
    df[f"{m}_why"] = [by_id.get(i, {}).get("why", "") for i in df.index]

T = [f"{m}_type" for m in MODELS]
C = [f"{m}_core" for m in MODELS]
df["type_agree"] = df[T].nunique(axis=1) == 1
df["core_votes"] = df[C].sum(axis=1)

print("claim_type counts")
print(pd.DataFrame({m: df[f"{m}_type"].value_counts() for m in MODELS}).reindex(TYPES).fillna(0).astype(int).to_string())
print("\ncore claims per model:", {m: int(df[f"{m}_core"].sum()) for m in MODELS})
print("core votes (0-3 models):", df.core_votes.value_counts().sort_index().to_dict())
print(f"\ntype agreement (all 3): {df.type_agree.mean():.0%}")
for a, b in [("haiku", "sonnet"), ("haiku", "opus"), ("sonnet", "opus")]:
    print(f"  {a} vs {b}: type {(df[f'{a}_type'] == df[f'{b}_type']).mean():.0%}, core {(df[f'{a}_core'] == df[f'{b}_core']).mean():.0%}")

claim_type counts
             haiku  sonnet  opus
OBSERVATION     20      26    25
ANALYSIS         4       2     5
VIEW            57      44    54
FORECAST        10      18    10
PREDICTION      25      25    22
ASSUMPTION       4       5     4

core claims per model: {'haiku': 3, 'sonnet': 5, 'opus': 4}
core votes (0-3 models): {0: 115, 1: 1, 2: 1, 3: 3}

type agreement (all 3): 78%
  haiku vs sonnet: type 82%, core 98%
  haiku vs opus: type 89%, core 99%
  sonnet vs opus: type 84%, core 99%


In [4]:
pd.set_option("display.max_colwidth", 100, "display.width", 250, "display.max_rows", 400)
# Propositions any model marked core; * = core vote
core = df[df.core_votes > 0].copy()
for m in MODELS:
    core[m] = core[f"{m}_type"].str[:4] + core[f"{m}_core"].map({True: "*", False: ""})
core[["text", "core_votes", *MODELS]]

,text,core_votes,haiku,sonnet,opus
P005,Fed is likely to hike 25bp.,3,PRED*,PRED*,PRED*
P007,Fed choice is simple: hike or bond spike.,2,VIEW,VIEW*,VIEW*
P009,"Warsh is likely to deliver hawkish hike, which should support higher front end rates and twist f...",3,PRED*,PRED*,PRED*
P010,"USD outcome will be determined by Fed ability to ""out-hawk"" mkt expectations and other G10 centr...",3,VIEW*,VIEW*,VIEW*
P085,The author expects Fed to deliver a hawkish hike.,1,PRED,PRED*,PRED


In [5]:
# Type disagreements
d = df.loc[~df.type_agree, ["text", *T]].copy()
d.columns = ["text", *MODELS]
d[MODELS] = d[MODELS].apply(lambda s: s.str[:4])
d

,text,haiku,sonnet,opus
P018,US 30Y could rapidly test 5.75%; UST buybacks would be futile to stop the UST long end move.,VIEW,FORE,VIEW
P019,"Bond price action, including post Friday's CPI, shows the bond likes orthodoxy (Exhibit 2).",ANAL,VIEW,ANAL
P020,Orthodoxy is use rate hikes to slow inflation and quell long end rate spike (not buybacks).,VIEW,OBSE,OBSE
P026,"If the author is right, UST curve is likely to twist flatten, USD is likely to be supported, and...",VIEW,FORE,VIEW
P037,The author sees a few potential tweaks to the FOMC statement.,PRED,VIEW,PRED
P044,"The Fed projections will also include policymakers' first estimates for 2029, which are likely t...",PRED,PRED,VIEW
P054,"The u-rate, however, is already two-tenths below the median projections for the end of this year...",ANAL,OBSE,ANAL
P055,So the author expects the median projections for the end of this year and next to move down as a...,VIEW,FORE,PRED
P060,If Warsh asserts that the Sep hike (if the Sep hike happens) was the first in a sequence of rate...,VIEW,FORE,VIEW
P066,The author will also be looking for additional color on Warsh's reaction function.,VIEW,OBSE,OBSE
